In [ ]:
# Clone repo 
import subprocess, sys, shutil, os
from kaggle_secrets import UserSecretsClient
clone_dir = "/kaggle/working/GA_STOMP.git"
# Use v6-spacing-regularity branch (now includes FFT reference-period fix)
if os.path.exists(clone_dir):
    shutil.rmtree(clone_dir)

subprocess.run([
    "git", "clone", "--quiet", "--branch", "v6-spacing-regularity",
    f"https://github.com/HoaPNG520/Motif-finding-GA-STOMP.git",
    clone_dir
], check=True)

print("Last commit:")
subprocess.run(["git", "-C", clone_dir, "log", "--oneline", "-3"])
print("Repository cloned successfully.")

In [ ]:
# P100 → use sm_60
# T4 → use sm_75
# P4 → use sm_61

In [ ]:
os.chdir(clone_dir)

# Check GPU first
!nvidia-smi --query-gpu=name --format=csv,noheader

# Build — adjust sm_60 if GPU is not P100
!nvcc -std=c++17 -arch=sm_60 -Iinclude -O3 -DNDEBUG \
  --expt-relaxed-constexpr --generate-line-info \
  -o ga_stomp \
  src/main.cu src/stomp.cu src/utils.cu src/fitness.cu \
  src/ga.cu src/io.cu src/timer.cu src/config.cu src/benchmark.cu

print("Build complete." if os.path.exists("ga_stomp") else "BUILD FAILED")

In [ ]:
import scipy.io, numpy as np
BASE_PATH = '/kaggle/input/datasets/sufian79/cwru-mat-full-dataset/'
files = {
    BASE_PATH+'105.mat': 'data/IR007_1797.csv',
    BASE_PATH+'106.mat': 'data/IR014_1797.csv',
    BASE_PATH+'130.mat': 'data/OR007_1797.csv',
    BASE_PATH+'100.mat': 'data/Normal.csv',
}

import os
os.makedirs('data', exist_ok=True)
os.makedirs('results/IR007', exist_ok=True)
os.makedirs('results/IR014', exist_ok=True)
os.makedirs('results/OR007', exist_ok=True)
os.makedirs('results/Normal', exist_ok=True)

for mat_path, csv_path in files.items():
    mat = scipy.io.loadmat(mat_path)
    key = [k for k in mat.keys() if 'DE_time' in k][0]
    signal = mat[key].flatten()[:6000]
    np.savetxt(csv_path, signal, fmt='%.8f')
    print(f'{mat_path.split("/")[-1]} → {csv_path}  key={key}')

In [ ]:
import subprocess

configs = [
    ('data/IR007_1797.csv', 'results/IR007', 'IR007 — Inner race true_w=74'),
    ('data/IR014_1797.csv', 'results/IR014', 'IR014 — Inner race true_w=74'),
    ('data/OR007_1797.csv', 'results/OR007', 'OR007 — Outer race true_w=112'),
    ('data/Normal.csv',     'results/Normal','Normal — no true window'),
]

# Complete config template with ALL keys
config_template = """
m_min=20
m_max=200
ez_min=0.25
ez_max=1.0
k_min=2
k_max=20
population=20
generations=20
tournament_k=3
mutation_rate=0.30
elite_count=2
approx_frac=0.5
n_seeds=3
verbose=1
input_path={input_path}
output_dir={output_dir}
"""

for input_path, output_dir, label in configs:
    print(f'\n{"="*60}')
    print(f'Running: {label}')
    print(f'{"="*60}')
    
    # Write config on the fly
    config = config_template.format(input_path=input_path, output_dir=output_dir)
    with open('run_config.ini', 'w') as f:
        f.write(config)
    
    result = subprocess.run(['./ga_stomp', 'run_config.ini'],
                          capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print('STDERR:', result.stderr)
    
    # Verify FFT period detection worked
    for line in result.stdout.split('\n'):
        if '[fft] Detected dominant period:' in line:
            print(f'>>> {line.strip()}')